# Phase 9: Robustness and Generalization Validation

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import warnings
from pathlib import Path
from itertools import product
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact, binomtest
import sys

#Trying statsmodels for McNemar
try:
    from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar
    STATSMODELS_AVAILABLE=True
except ImportError:
    STATSMODELS_AVAILABLE=False

warnings.filterwarnings('default')
np.random.seed(36)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

#Paths
PROJECT_ROOT=Path.cwd().parent
PHASE6_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase6_models'
PHASE8_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase8_structural_analysis'
PHASE9_OUTPUT=PROJECT_ROOT / 'data' / 'processed' / 'phase9_robustness'
PHASE9_OUTPUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0,str(PROJECT_ROOT))        #To access util functions from src

#Constants
FDR_ALPHA=0.05
EFFECT_THRESHOLD_CONTINUOUS=0.10
EFFECT_THRESHOLD_FAILURE=0.01
MIN_SAMPLE_WARNING=10

DATASETS=['2007', '2008']
K_VALUES=[1, 3, 5, 10]
MODELS=['pointwise', 'pairwise', 'lightgbm']
PIPELINES=['raw', 'global', 'per_query']

#Baseline
BASELINE_MODEL='pointwise'
BASELINE_PIPELINE='raw'

warnings_issued=[]

print('='*80)
print('PHASE 9: ROBUSTNESS & GENERALIZATION VALIDATION')
print('='*80)
print(f'Output: {PHASE9_OUTPUT}')
print(f'Baseline: {BASELINE_MODEL}_{BASELINE_PIPELINE}')
print(f'McNemar: {"statsmodels" if STATSMODELS_AVAILABLE else "manual fallback"}')
print('='*80)

PHASE 9: ROBUSTNESS & GENERALIZATION VALIDATION
Output: b:\Arlington\Arlington\4th_sem\Capstone\Capstone_coding\data\processed\phase9_robustness
Baseline: pointwise_raw
McNemar: statsmodels


## 2. Utility Functions

In [9]:
from src.utils import (
    make_fail_flag,
    bh_fdr,
    safe_k,
    check_sample_size,
    compute_failure_at_k,
    mcnemar_test
)

print("Utilities imported from src.ltr_utils")

Utilities imported from src.ltr_utils


## 3. Loading Phase 6 artifacts

In [10]:
print('\n'+'='*80)
print('LOADING PHASE 6 ARTIFACTS')
print('='*80)

def load_phase6_artifacts(dataset: str) -> dict:
    #Load Phase 6 with strict validation.
    artifacts={'query_metrics': {}, 'predictions': {}}
    for model, pipeline in product(MODELS, PIPELINES):
        key=f'{model}_{pipeline}_{dataset}'
        qm_file=PHASE6_OUTPUT / f'{key}_query_metrics.csv'
        pred_file=PHASE6_OUTPUT / f'{key}_predictions.csv'
        
        if not qm_file.exists():
            raise RuntimeError(f'MISSING: {qm_file}')
        if not pred_file.exists():
            raise RuntimeError(f'MISSING: {pred_file}')
        
        qm=pd.read_csv(qm_file)
        pred=pd.read_csv(pred_file)
        
        #Validating columns
        for col in ['qid', 'num_docs', 'num_relevant_1', 'Failure@5_primary']:
            if col not in qm.columns:
                raise RuntimeError(f'Missing "{col}" in {qm_file}')
        for col in ['qid', 'label', 'score']:
            if col not in pred.columns:
                raise RuntimeError(f'Missing "{col}" in {pred_file}')
        
        #Coercing dtypes
        qm['qid']=qm['qid'].astype(int)
        pred['qid']=pred['qid'].astype(int)
        pred['label']=pred['label'].astype(int)
        pred['score']=pred['score'].astype(float)
        
        artifacts['query_metrics'][key]=qm
        artifacts['predictions'][key]=pred
        print(f'{key:35s} ({len(qm)} queries, {len(pred)} docs)')
    return artifacts

artifacts_2007=load_phase6_artifacts('2007')
artifacts_2008=load_phase6_artifacts('2008')
all_artifacts={'2007': artifacts_2007, '2008': artifacts_2008}

print(f'\nLoaded artifacts for {len(DATASETS)} datasets')
print('='*80)


LOADING PHASE 6 ARTIFACTS
pointwise_raw_2007                  (336 queries, 13652 docs)
pointwise_global_2007               (336 queries, 13652 docs)
pointwise_per_query_2007            (336 queries, 13652 docs)
pairwise_raw_2007                   (336 queries, 13652 docs)
pairwise_global_2007                (336 queries, 13652 docs)
pairwise_per_query_2007             (336 queries, 13652 docs)
lightgbm_raw_2007                   (336 queries, 13652 docs)
lightgbm_global_2007                (336 queries, 13652 docs)
lightgbm_per_query_2007             (336 queries, 13652 docs)
pointwise_raw_2008                  (156 queries, 2874 docs)
pointwise_global_2008               (156 queries, 2874 docs)
pointwise_per_query_2008            (156 queries, 2874 docs)
pairwise_raw_2008                   (156 queries, 2874 docs)
pairwise_global_2008                (156 queries, 2874 docs)
pairwise_per_query_2008             (156 queries, 2874 docs)
lightgbm_raw_2008                   (156 queries,